In [1]:
import sys
import os

import torch
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
from typing import Any, NamedTuple

from mllib import rllib

In [2]:
import gym_simpletetris
class ClipRewardEnv(gym.RewardWrapper):
    """
    Clips the reward to {+1, 0, -1} by its sign.
    Args:
        env (gym.Env): The environment to wrap
    """

    def __init__(self, env: gym.Env):
        gym.RewardWrapper.__init__(self, env)
    
    def reward(self, reward: float) -> float:
        return np.sign(reward)

np.float = np.float32
envname = 'BreakoutNoFrameskip-v4'
def create_env(mode='rgb_array'):
    env =  gym.wrappers.AtariPreprocessing(
        gym.make(envname, render_mode='rgb_array'),
        noop_max=30,                   # 30 random actions a the beginning of an episode
        frame_skip=4,
        screen_size=84,                # Changes observation size to 84x84
        terminal_on_life_loss=True,    # Returns done=True if episode terminates
        grayscale_obs=True,            # Convert RGB to grayscale
        scale_obs=True,                # Scales observations to range 0-1
    )
    env = ClipRewardEnv(env)                           # Clip all rewards to {-1, 0, +1}
    env = gym.wrappers.FrameStack(env, 4) 
    env = rllib.TorchObservationWrapper(env)
    return env

In [3]:
env = create_env()

A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


In [4]:
N = 256

class Model(nn.Module):
    def __init__(self, act_size):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64*7*7,N),
            nn.ReLU(),
            nn.Linear(N, act_size),
        )
    
    def forward(self, x):
        if x.ndim == 3:
            x = torch.reshape(x, (-1,) + x.shape)
        return self.net(x)


In [5]:
import torch
from torch.utils.tensorboard import SummaryWriter
from ray import tune
from ray.tune.search import ConcurrencyLimiter

import ray
from ray.tune.search.hyperopt import HyperOptSearch
space = {
    'lr': tune.choice([1e-3,2.5e-4,1e-4]),
    'gamma': tune.choice([0.99]),
    'replay_buf_size': tune.choice([10000]),
    'batch_size': tune.choice([32,64,128]),
    'train_interval': tune.choice([4, 8, 16, 64]),
    'train_count': tune.choice([1]),
    'update_interval': tune.choice([5000, 10000, 20000]),
    'eps_decay': tune.choice([3e5, 5e5, 7e5]),
}

In [6]:
config = {
    'lr': 1e-4,
    'gamma': 0.99,
    'replay_buf_size': 60000,
    'batch_size': 32,
    'train_interval': 8,
    'train_count': 2,
    'update_interval': 10000,
    'eps_decay': 3e5,
}

In [7]:
# env.observation_space
num_acts = env.action_space.n
model = Model(num_acts)
# model.load_state_dict(torch.load("breakout.model"))

In [ ]:
logdir = '/home/sunho/dev/MLStudy/RL/dqn/runs/'

cuda = torch.device('cuda')
ri = 0
li = 0
def objective(config):
    global ri
    global li
    ri = 0
    li = 0
    suffix = "lr={},gamma={},replay_buf_size={},batch_size={},train_count={},update_interval={},eps_decay={},train_interval={}".format(
        config["lr"], config["gamma"], config["replay_buf_size"], config["batch_size"], config["train_count"], 
        config["update_interval"], config["eps_decay"], config['train_interval'])
    # writer = SummaryWriter(logdir+suffix)
    writer = SummaryWriter()
    def create_model():
        return Model(num_acts)

    agent = rllib.DQNAgent(model, create_model(), cuda, env.action_space, 1.0, 0.1, config["eps_decay"])
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    options = rllib.DQNOptions(optimizer)
    options.replay_buf_size = config["replay_buf_size"]
    options.num_steps = 5000000
    options.double_dqn = True
    options.max_episode_len = 10000
    options.gamma = config["gamma"]
    options.update_interval = config["update_interval"]
    options.batch_size = config["batch_size"]
    options.train_count =  config["train_count"]
    options.train_interval = config["train_interval"]
    
    def add_reward(x):
        global ri
        writer.add_scalar("Reward/train", x, ri)
        ri += 1
    
    def add_loss(x):
        global li
        for k,v in x.items():
            writer.add_scalar("Loss/" + k, v, li)
        li += 1
    
    options.report_reward = add_reward
    options.report_train = add_loss
    stats = rllib.dqn_train(create_env, agent, cuda, options)
    torch.save(model.state_dict(), "breakout5.model")
    score = np.mean(stats.reward_history[-128:])
    return {"score": score, "model": model, "agent": agent, "optimizer": optimizer}
res = objective(config)

/opt/miniconda3/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [ ]:
optimizer = res["optimizer"]
model = res["model"]
agent = res["agent"]

In [16]:
algo = HyperOptSearch()
algo = ConcurrencyLimiter(algo, max_concurrent=4)
tuner = tune.Tuner(
    objective,
    tune_config=tune.TuneConfig(
        metric="score",
        mode="max",
        search_alg=algo,
        num_samples=16,
    ),
    param_space=space,
)
res = tuner.fit()

(pid=951187) A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
(pid=951187) [Powered by Stella]
(objective pid=951187) /opt/miniconda3/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
(objective pid=951187)   if not isinstance(terminated, (bool, np.bool8)):
(pid=971917) A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7) [repeated 4x across cluster]
(pid=971917) [Powered by Stella] [repeated 4x across cluster]
(objective pid=951474) /opt/miniconda3/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24) [repeated 3x across cluster]
(objective pid=951474)   if not isinstance(terminated, (bool, np.bool8)): [repeated 3x across cluster]
(pid=1016443) A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
(pid=1016443) [Powered by Stella]
(objective pid=9719

In [9]:
model = res["model"]

In [16]:
model = model.cpu()
torch.save(model.state_dict(), "breakout3.model")


In [14]:
agent = rllib.DQNAgent(model, model, None, env.action_space, 0.00, 0.00, 500000)
rllib.run_jupyter(create_env, agent, None, 1000000)

KeyboardInterrupt: 